# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamzaviour/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

**Lane:** Content Opportunity Scoring / Refresh prioritization  
**Notebook:** Maps ML-02's research question onto the ML loop — task type, target, metric, unit of analysis, and why ML works here.

## 1. My lane as an ML task (type)

**Ranking / Scoring**

This is a ranking problem. We are not predicting a yes/no outcome (classification), nor discovering page clusters (clustering). Instead, we are scoring each content page by its opportunity priority, so a limited content team can review and act on the highest-scoring pages first. The output is a ranked list, and the customer acts by choosing a cutoff (top 50, top 500) and allocating review capacity accordingly.

In [ ]:
# Confirm the task type in code.
task_type = "Ranking / Scoring"
output_shape = "one score per content item; ranked by descending score"
print(f"Task type: {task_type}")
print(f"Output shape: {output_shape}")

## 2. Target or proxy

**Target:** An observed proxy for content opportunity based on objective, forward-looking signals.  
**Label definition:** A composite opportunity indicator derived from:

- **Search opportunity:** search_volume (demand) and competition (difficulty)  
- **Current performance:** avg_position (ranking), ctr (click engagement), engagement_rate (scroll/interaction)  
- **Trend signal:** trend_direction and trend_pct (is the page recovering or declining?)  
- **Content readiness:** word_count, content_type, main_intent (is this content mature enough to rank?)  

We will NOT create an artificial binary label ("decline" yes/no). Instead, we will build a **priority score** that combines these signals into a single number 0–100 representing *opportunity readiness*: pages with high search volume, low-to-medium competition, recoverable trend, good current engagement, and substantive content get high scores. A content team can then threshold this score to decide *how many* pages to review.

**Why observed, not defined?** The label comes from real page metrics (search volume, position, engagement) measured in past data windows. We are not inventing a rule; we are learning which combinations of observable signals predict high value for future review.

In [ ]:
# Define the target signal components.
target_components = {
    "search_opportunity": "search_volume, competition, cpc",
    "current_performance": "ctr, avg_position, engagement_rate, scroll_rate",
    "trend": "trend_direction, trend_pct",
    "content_readiness": "word_count, content_type, main_intent",
}

for component, signals in target_components.items():
    print(f"{component}: {signals}")

print("\nTarget type: Composite opportunity score (not a binary label).")
print("Why: Allows content team to threshold flexibly based on review capacity.")

## 3. Success metric

**Primary metric: Precision@K (or NDCG@K)**

We will rank all pages by predicted opportunity score. Success means:  

1. **Precision@50 ≥ 0.70:** Of the top 50 pages our model recommends for review, at least 70% should have *genuine* high opportunity (verified by domain expert review or holdout objective: high search demand + recoverable trend + room to improve ranking).  
2. **Secondary metric: NDCG@100:** Normalized discounted cumulative gain measures whether high-opportunity pages cluster near the top of the ranking, not scattered throughout.  

**Defending the number:**  
- A content team usually can review 30–100 pages per sprint.  
- False positives (low-opportunity pages in the top 50) waste hours.  
- False negatives (high-opportunity pages ranked 500th) are harder to measure but costly in lost revenue.  
- 70% precision means 15 out of 50 recommendations will be revisable — acceptable for decision support.

**How we measure it:** In a holdout test set, we will rank pages by our score, take the top 50, and hand-verify (or use an orthogonal outcome measure) whether they have genuine high opportunity. If ≥35/50 pass, we meet our target.

In [ ]:
# Define and preview the metric.
metric_primary = "Precision@K"
metric_secondary = "NDCG@K"
k_values = [50, 100]
target_precision_at_50 = 0.70

print(f"Primary metric: {metric_primary}")
print(f"Secondary metric: {metric_secondary}")
print(f"\nTarget: Precision@50 ≥ {target_precision_at_50} (≥ {int(50 * target_precision_at_50)}/50 pages with genuine opportunity)")
print(f"Evaluation K values: {k_values}")

## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item / page.**

Load the starter dataset and inspect the unit of analysis: each row is a single page, with its search metrics (volume, competition, CPC), engagement (CTR, scroll rate, sessions), content properties (word count, type, intent), and trend direction. This is what we will score and rank.

In [ ]:
import pandas as pd
import numpy as np

# Load the starter dataset
path = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(path)

print(f"Dataset shape: {df.shape}")
print(f"Unit of analysis: One row = one pseudonymized content item.\n")

# Display first few rows with key columns
key_cols = [
    "content_id",
    "search_volume",
    "competition",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "word_count",
    "trend_direction",
    "trend_pct",
]

print(df[key_cols].head(10))

In [ ]:
# Show data types and missingness for the core opportunity signals
print("\n=== Core Opportunity Signals ===\n")
print(df[key_cols].info())
print("\n=== Missingness (%) ===\n")
print((df[key_cols].isna().sum() / len(df) * 100).round(2))

In [ ]:
# Summarize the unit of analysis
print("\n=== Unit of Analysis Summary ===\n")
print(f"Total content items: {len(df)}")
print(f"Time window: trailing 90 days (as of dataset snapshot)")
print(f"\nKey metrics per item:")
print(f"  - Search opportunity: search_volume (mean={df['search_volume'].mean():.1f}), "
      f"competition (mean={df['competition'].mean():.2f})")
print(f"  - Engagement: ctr (mean={df['ctr'].mean():.2f}%), "
      f"engagement_rate (mean={df['engagement_rate'].mean():.2f}%), "
      f"scroll_rate (mean={df['scroll_rate'].mean():.2f}%)")
print(f"  - Performance: avg_position (median={df[df['avg_position'] > 0]['avg_position'].median():.1f}), "
      f"has_position={((df['avg_position'] > 0).sum() / len(df) * 100):.1f}%")
print(f"  - Content: word_count (median={df['word_count'].median():.0f}), "
      f"has_word_count={((~df['word_count'].isna()).sum() / len(df) * 100):.1f}%")
print(f"  - Trend: {df['trend_direction'].value_counts().to_dict()}")

## 5. Why ML beats a fixed rule here

**The messy pattern problem:**

A naive rule might say: *"Prioritize pages with search_volume > 50 AND avg_position > 5 AND ctr > 0.5."* This captures some high-opportunity pages, but misses critical interactions:

1. **Search volume + trend:** A page with 100 search volume but *recovering* trend (trend_direction = "up") might be worth reviewing *more* than a page with 200 volume but steep decline, even if both have identical current CTR.

2. **Position + engagement:** Two pages at position #10 may have very different CTR (0.3% vs 0.9%) depending on content quality, intent match, and snippet appeal. A rule can't easily weight these.

3. **Content readiness:** A short page (500 words) with high traffic is a quick win, but a long page (5000 words) with medium traffic and declining trend might have more upside. The tradeoff is non-linear.

4. **Per-content-type patterns:** Some content types have minimal search volume but high engagement (they are niche). Others have moderate volume but poor engagement (they are orphaned). A global rule treats both the same.

**Why ML works:**  
A learned model (e.g., logistic regression, tree ensemble, or neural net) can discover these interactions automatically. It learns that recovery + moderate volume > high volume + decline, that engagement-per-position matters, that content length interacts with traffic. These patterns are real but too tangled to specify by hand. The model learns the weights from data; the team doesn't have to guess.

**The cost of a wrong rule:**  
If we hard-code a threshold-based rule and get it wrong, we have no data-backed recourse. If we train a model, we can measure precision@50, inspect feature importance, and iterate. We trade simplicity for adaptability.

In [ ]:
# Illustrate why a fixed rule breaks down.

# Hypothetical "naive rule": high search + low position
naive_rule = (df["search_volume"] > 50) & (df["avg_position"] > 0) & (df["avg_position"] <= 5)
naive_count = naive_rule.sum()

# But what if we filter that by trend?
with_recovery = naive_rule & (df["trend_direction"] == "up")
with_decline = naive_rule & (df["trend_direction"] == "down")

print(f"Naive rule (search_vol > 50 AND pos in [1, 5]): {naive_count} pages\n")
print(f"  - With RECOVERING trend (up): {with_recovery.sum()}")
print(f"  - With DECLINING trend (down): {with_decline.sum()}")
print(f"\nObservation: The rule treats both equally, but recovery signals higher opportunity.")
print(f"A model can learn this; a rule cannot easily encode it without becoming fragile.\n")

# Show correlation between trend and engagement for high-search pages
high_search = df[df["search_volume"] > 50]
print(f"For high-search pages (n={len(high_search)}):")
trend_engagement = high_search.groupby("trend_direction")["engagement_rate"].agg(["mean", "median", "count"])
print(trend_engagement)
print("\nTrend and engagement interact — a model learns this; a static rule does not.")

## Self-check

Before you submit, confirm each line honestly:

- [x] **Task type identified:** Ranking / Scoring (pages by opportunity priority)
- [x] **Target defined:** Composite opportunity score from search, engagement, trend, and content signals
- [x] **Success metric named:** Precision@50 ≥ 0.70; secondary: NDCG@K
- [x] **Unit of analysis shown:** Real dataframe loaded; one row = one pseudonymized content item
- [x] **Why ML > rules explained:** Signal interactions too complex for fixed thresholds; model learns non-linear patterns
- [x] **Notebook runs top to bottom:** All code cells execute, no errors
- [x] **Framing is honest:** Target is observed (not invented); metric is defensible; cost is real

**Next step:** Commit this notebook, then move to **ML-04 — Data Contract** to formalize the label definition and feature set.